# Hyper Param Cost Violins

- hyper params
  - population + M_offspring * N_crossover
  - N_mut * W * D * gen
  - population
  - generations

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

import pandas as pd
import geopandas as gpd
import numpy as np
from matplotlib import pyplot as plt
import cartopy
import cmocean
import xarray as xr
import seaborn as sns

from load_tuning_results import (
    load_results,
    get_forcing_df,
    get_seed_routes_gdf,
    filter_suspicious_routes,
    add_derived_features,
)

import warnings

warnings.filterwarnings("ignore")

In [ ]:
# parameters

# results dataframe from
gpq_file = "../results/results_prelim.geoparquet"

In [ ]:
gdf = gpd.read_parquet(gpq_file)
gdf = add_derived_features(gdf)
gdf = filter_suspicious_routes(gdf)
gdf

In [ ]:
_gdf = gdf.where(
    (gdf.hyper_hazard_penalty_multiplier == 0) & (gdf.journey_speed_knots == 12.0)
).dropna(how="all")

In [ ]:
_gdf = _gdf.assign(
    hyper_mu_plus_lambda=(
        _gdf.hyper_population_size
        + _gdf.hyper_offspring_size * _gdf.hyper_crossover_rounds
    ).astype("category"),
    hyper_mut_step_size=(
        _gdf.hyper_mutation_iterations
        * _gdf.hyper_mutation_width_fraction
        * _gdf.hyper_mutation_displacement_fraction
        * _gdf.hyper_generations.astype(int)
    ).astype("category"),
    hyper_mutation_iterations=_gdf.hyper_mutation_iterations.astype("category"),
    hyper_crossover_rounds=_gdf.hyper_crossover_rounds.astype("category"),
)

In [ ]:
_gdf.elite_cost_relative.max()

In [ ]:
__gdf = _gdf[
    _gdf["elite_cost_relative"]
    <= _gdf.groupby(
        [
            "hyper_generations",
            "hyper_population_size",
            "forcing_scenario_name",
            "hyper_enable_adaptation",
        ]
    )["elite_cost_relative"].transform(lambda x: x.quantile(0.99))
]

In [ ]:
sns.set_context("paper", font_scale=2.0)
sns.set_style("whitegrid")
g = sns.catplot(
    data=__gdf,
    y="elite_cost_relative",
    x="hyper_generations",
    # x="journey_name",
    col="hyper_population_size",
    row="forcing_scenario_name",
    # hue="hyper_enable_adaptation",
    hue="hyper_mu_plus_lambda",
    kind="violin",
    cut=0,
    inner="quartiles",
    # palette="Dark2",
    height=4.0,
    aspect=1,
    margin_titles=True,
)
g.set_titles(col_template="population {col_name}", row_template="{row_name}")
sns.move_legend(g, bbox_to_anchor=(0.85, 0.26), loc="lower right", ncol=6, frameon=True)
g.fig.suptitle("No Hazards, 12 knots")
g.fig.tight_layout()
g.fig.savefig("../figures/030_hyper_param_cost_violins.pdf", dpi=200)
g.fig.savefig("../figures/030_hyper_param_cost_violins.png", dpi=200)

In [ ]:
sns.set_context("paper", font_scale=2.0)
sns.set_style("whitegrid")
g = sns.catplot(
    data=__gdf,
    y="runtime_seconds",
    # hue="hyper_generations",
    x="hyper_mu_plus_lambda",
    hue="hyper_population_size",
    kind="box",
    palette="Dark2",
    aspect=2,
    margin_titles=True,
)
g.fig.suptitle("No Hazards, 12 knots")

g.fig.savefig("../figures/030_hyper_param_runtime_boxes.pdf", dpi=200)
g.fig.savefig("../figures/030_hyper_param_runtime_boxes.png", dpi=200)